# TD Mini-Projet B — Détection de Deepfakes
**M1 IA & Cybersécurité — Cours IA & Sécurité**

---

## Contexte

Les deepfakes — images ou vidéos générées par IA — représentent une menace croissante : désinformation, usurpation d'identité, fraude documentaire. Dans le secteur aéronautique, de fausses images de certificats, de pièces ou de rapports d'inspection pourraient avoir des conséquences graves.

Dans ce projet, vous allez entraîner un classifieur capable de distinguer un visage réel d'un visage généré par IA, puis analyser ses erreurs pour comprendre les limites de la détection automatique.

---

## ⚠️ Avant de commencer — Attacher le dataset

Ce notebook tourne sur **Kaggle**. Le dataset doit être attaché avant de lancer les cellules.

**Étapes :**
1. Dans le panneau de droite, cliquez sur **"+ Add Input"**
2. Cliquez sur l'onglet **"Datasets"**
3. Recherchez : `140k-real-and-fake-faces`
4. Cliquez le **"+"** à côté de **"140k Real and Fake Faces"** (auteur : xhlulu)
5. Le dataset apparaît sous "Input" → vous pouvez fermer le panneau

---

## Objectifs
1. Vérifier l'environnement et le dataset
2. Explorer les données visuellement
3. Entraîner un classifieur par transfer learning (EfficientNet-B0)
4. Évaluer les performances
5. Analyser les erreurs du modèle


---
## Étape 0 — Vérification de l'environnement

**Lancez cette cellule en premier.** Elle vérifie que le dataset est bien attaché et que le GPU est disponible. Si elle affiche une erreur, le dataset n'est pas attaché — suivez les instructions ci-dessus.

In [ ]:
import os
import torch

# Vérification GPU
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Dispositif : {DEVICE}')
if DEVICE.type == 'cpu':
    print('⚠️  Pas de GPU détecté. Activez-le : Settings → Accelerator → GPU T4 x2')
else:
    print('✅ GPU disponible — entraînement ~15 min pour 3 époques')

# Vérification dataset
DATA_DIR = '/kaggle/input/datasets/xhlulu/140k-real-and-fake-faces/real_vs_fake/real-vs-fake'

if not os.path.exists(DATA_DIR):
    print('\n❌ Dataset introuvable. Vérifiez que "140k Real and Fake Faces" est bien attaché dans le panneau Input.')
else:
    TRAIN_DIR = os.path.join(DATA_DIR, 'train')
    VALID_DIR = os.path.join(DATA_DIR, 'valid')
    TEST_DIR  = os.path.join(DATA_DIR, 'test')
    print('\n✅ Dataset trouvé :')
    for split, path in [('train', TRAIN_DIR), ('valid', VALID_DIR), ('test', TEST_DIR)]:
        reels = len(os.listdir(os.path.join(path, 'real')))
        fakes = len(os.listdir(os.path.join(path, 'fake')))
        print(f'  {split:6s} — réels : {reels:6d} | fakes : {fakes:6d}')

---
## Étape 1 — Imports

In [ ]:
import random
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.image as mpimg

import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms, models
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

print('✅ Imports OK')

---
## Étape 2 — Exploration visuelle

> 💡 **Analogie** : Avant de former un pilote à repérer une panne, on lui montre des centaines d'exemples. Même logique ici : regardez les données avant d'entraîner quoi que ce soit.

Pouvez-vous repérer des différences visuelles entre images réelles et générées par IA ?

In [ ]:
def afficher_exemples(dossier_reel, dossier_fake, n=4):
    fig, axes = plt.subplots(2, n, figsize=(3*n, 6))
    fig.suptitle('Visages réels (haut) vs générés par IA (bas)', fontsize=13)

    fichiers_reels = random.sample(os.listdir(dossier_reel), n)
    for i, f in enumerate(fichiers_reels):
        img = mpimg.imread(os.path.join(dossier_reel, f))
        axes[0, i].imshow(img)
        axes[0, i].axis('off')
        axes[0, i].set_title('Réel', fontsize=10, color='green')

    # À vous : même logique pour les fakes
    # Indice : remplacer dossier_reel par dossier_fake
    fichiers_fakes = ???
    for i, f in enumerate(fichiers_fakes):
        img = ???
        axes[1, i].imshow(img)
        axes[1, i].axis('off')
        axes[1, i].set_title('Fake', fontsize=10, color='red')

    plt.tight_layout()
    plt.show()

afficher_exemples(
    os.path.join(TRAIN_DIR, 'real'),
    os.path.join(TRAIN_DIR, 'fake')
)

**✏️ Observation** : notez ici ce que vous observez (artefacts, flou, symétrie, arrière-plans...) :

???

---
## Étape 3 — Préparation des données

> 💡 EfficientNet-B0 attend des images 224×224 normalisées avec les valeurs ImageNet standard.

In [ ]:
MOYENNE    = [0.485, 0.456, 0.406]
ECART_TYPE = [0.229, 0.224, 0.225]

# Transformations entraînement (avec augmentation)
transform_train = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ColorJitter(brightness=0.2),
    transforms.ToTensor(),
    transforms.Normalize(mean=MOYENNE, std=ECART_TYPE)
])

# À vous : définir transform_val SANS augmentation
# (uniquement Resize, ToTensor, Normalize)
transform_val = transforms.Compose([
    ???
])

print('✅ Transformations définies.')

In [ ]:
BATCH_SIZE = 64

dataset_train = datasets.ImageFolder(TRAIN_DIR, transform=transform_train)
dataset_val   = datasets.ImageFolder(VALID_DIR, transform=transform_val)
dataset_test  = datasets.ImageFolder(TEST_DIR,  transform=transform_val)

loader_train = DataLoader(dataset_train, batch_size=BATCH_SIZE, shuffle=True,  num_workers=2)
# À vous : créer loader_val et loader_test (shuffle=False)
loader_val   = DataLoader(???)
loader_test  = DataLoader(???)

print(f'Classes : {dataset_train.classes}')
print(f'Train : {len(dataset_train)} | Val : {len(dataset_val)} | Test : {len(dataset_test)}')

---
## Étape 4 — Construction du modèle

> 💡 **Transfer learning** : EfficientNet-B0 a appris à reconnaître des formes et textures sur des millions d'images. On garde ses couches de base et on remplace uniquement la dernière couche pour notre tâche (réel / fake).

In [ ]:
def creer_modele():
    modele = models.efficientnet_b0(weights=models.EfficientNet_B0_Weights.IMAGENET1K_V1)

    # Geler toutes les couches (on ne réentraîne pas le tronc)
    for param in modele.parameters():
        param.requires_grad = False

    # À vous : remplacer modele.classifier par Sequential(
    #   Dropout(0.3) + Linear(1280, 2)
    # )
    modele.classifier = nn.Sequential(
        ???
    )
    return modele.to(DEVICE)

modele = creer_modele()
params_entrainables = sum(p.numel() for p in modele.parameters() if p.requires_grad)
params_total        = sum(p.numel() for p in modele.parameters())
print(f'Paramètres entraînables : {params_entrainables:,} / {params_total:,}')

In [ ]:
critere = nn.CrossEntropyLoss()

# À vous : créer optimiseur Adam, lr=0.001
# uniquement sur les paramètres requires_grad=True
optimiseur = optim.Adam(
    ???,
    lr=0.001
)
print('✅ Critère et optimiseur définis.')

---
## Étape 5 — Entraînement

> ⏱️ **Durée estimée sur GPU Kaggle : ~15 min pour 3 époques.** Assurez-vous que le GPU est activé (Settings → Accelerator → GPU T4 x2) avant de lancer.

In [ ]:
def entrainer_une_epoque(modele, loader, critere, optimiseur):
    modele.train()
    perte_totale = 0.0
    corrects = 0

    for images, etiquettes in loader:
        images, etiquettes = images.to(DEVICE), etiquettes.to(DEVICE)
        optimiseur.zero_grad()
        sorties = modele(images)
        perte = critere(sorties, etiquettes)

        # À vous : rétropropagation + mise à jour des poids
        ???
        ???

        perte_totale += perte.item() * images.size(0)
        _, predictions = torch.max(sorties, 1)
        corrects += (predictions == etiquettes).sum().item()

    return perte_totale / len(loader.dataset), corrects / len(loader.dataset)


def evaluer(modele, loader, critere):
    modele.eval()
    perte_totale = 0.0
    corrects = 0
    with torch.no_grad():
        for images, etiquettes in loader:
            images, etiquettes = images.to(DEVICE), etiquettes.to(DEVICE)
            sorties = modele(images)
            perte = critere(sorties, etiquettes)
            perte_totale += perte.item() * images.size(0)
            _, predictions = torch.max(sorties, 1)
            corrects += (predictions == etiquettes).sum().item()
    return perte_totale / len(loader.dataset), corrects / len(loader.dataset)

In [ ]:
NB_EPOQUES = 3
historique = {'perte_train': [], 'acc_train': [], 'perte_val': [], 'acc_val': []}

for epoque in range(1, NB_EPOQUES + 1):
    perte_tr, acc_tr = entrainer_une_epoque(modele, loader_train, critere, optimiseur)
    perte_v,  acc_v  = evaluer(modele, loader_val, critere)

    historique['perte_train'].append(perte_tr)
    historique['acc_train'].append(acc_tr)
    historique['perte_val'].append(perte_v)
    historique['acc_val'].append(acc_v)

    print(f'Époque {epoque}/{NB_EPOQUES} '
          f'| Train — perte : {perte_tr:.4f}  acc : {acc_tr:.3f} '
          f'| Val   — perte : {perte_v:.4f}  acc : {acc_v:.3f}')

print('\n✅ Entraînement terminé.')

In [ ]:
# Courbes d'apprentissage
epoques = range(1, NB_EPOQUES + 1)
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(epoques, historique['perte_train'], label='Train', marker='o')
axes[0].plot(epoques, historique['perte_val'],   label='Validation', marker='o')
axes[0].set_title('Perte'); axes[0].legend()

# À vous : tracer les courbes d'accuracy (même structure)
axes[1].plot(???)
axes[1].plot(???)
axes[1].set_title("Accuracy"); axes[1].legend()

plt.tight_layout()
plt.show()

---
## Étape 6 — Évaluation sur le jeu de test

In [ ]:
def obtenir_predictions(modele, loader):
    modele.eval()
    toutes_pred = []
    toutes_etiq = []
    with torch.no_grad():
        for images, etiquettes in loader:
            images = images.to(DEVICE)
            sorties = modele(images)
            _, predictions = torch.max(sorties, 1)
            toutes_pred.extend(predictions.cpu().numpy())
            toutes_etiq.extend(etiquettes.numpy())
    return np.array(toutes_etiq), np.array(toutes_pred)

y_test, y_pred = obtenir_predictions(modele, loader_test)

print('=== Rapport de classification ===')
print(classification_report(y_test, y_pred, target_names=['fake', 'real']))

In [ ]:
mc = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(6, 5))
# À vous : sns.heatmap avec annot=True, fmt='d', cmap='Blues'
# xticklabels et yticklabels : ['fake', 'real']
sns.heatmap(???)
plt.title('Matrice de confusion — Test')
plt.ylabel('Étiquette réelle')
plt.xlabel('Prédiction')
plt.show()

---
## Étape 7 — Analyse des erreurs

> 💡 C'est l'étape la plus importante pédagogiquement. Comprendre pourquoi le modèle se trompe est plus utile que d'améliorer son score.

In [ ]:
def denormaliser(tensor):
    moyenne   = torch.tensor(MOYENNE).view(3, 1, 1)
    ecart_type = torch.tensor(ECART_TYPE).view(3, 1, 1)
    img = tensor.cpu() * ecart_type + moyenne
    return img.permute(1, 2, 0).numpy().clip(0, 1)


def trouver_erreurs(modele, dataset, n=8):
    modele.eval()
    erreurs = []
    with torch.no_grad():
        for img, etiquette in dataset:
            if len(erreurs) >= n:
                break
            sortie = modele(img.unsqueeze(0).to(DEVICE))
            _, pred = torch.max(sortie, 1)
            pred = pred.item()
            # À vous : ajouter à erreurs si prédiction != étiquette
            if ???:
                erreurs.append((img, etiquette, pred))
    return erreurs


erreurs = trouver_erreurs(modele, dataset_test, n=8)
noms = dataset_test.classes

fig, axes = plt.subplots(2, 4, figsize=(14, 7))
fig.suptitle('Images mal classifiées', fontsize=13)
for ax, (img, vrai, predit) in zip(axes.flat, erreurs):
    ax.imshow(denormaliser(img))
    ax.set_title(f'Vrai : {noms[vrai]}\nPrédit : {noms[predit]}',
                 fontsize=9, color='red')
    ax.axis('off')
plt.tight_layout()
plt.show()

---
## Étape 8 — Analyse et restitution

Répondez aux questions suivantes pour préparer votre présentation.

**1. Métriques sur le jeu de test :**

- Accuracy : ???
- Précision (fake) : ???
- Rappel (fake) : ???
- F1-score (fake) : ???

**2. En regardant les images mal classifiées, qu'est-ce qui semble tromper le modèle ?**

???

**3. Y a-t-il plus de faux positifs (réels classés fake) ou de faux négatifs (fakes classés réels) ? Quelle erreur est la plus grave dans un contexte de sécurité ?**

???

**4. Ce modèle serait-il déployable dans un système de vérification documentaire aéronautique ? Justifiez.**

???

**5. Que faudrait-il améliorer pour le rendre plus robuste ?**

???

---
## Trame de restitution

**Titre** : Détection de Deepfakes — Résultats

**Ce que nous avons fait** *(3 lignes max)* : ???

**Le résultat le plus surprenant** : ???

**1 leçon retenue pour la pratique professionnelle** : ???

**Démo live** : afficher les images mal classifiées et expliquer pourquoi le modèle échoue.